# Credit Card Fraud Detection with Benford's Law

Multi-model comparison on the ULB credit card fraud dataset. Inspired by Stripe's Payments Foundation Model (sequence-aware fraud detection) and classical forensic accounting (Benford's Law).

**Run All** on Colab to reproduce.

## 0. Bootstrap

In [ ]:
import os, sys

REPO_URL = "https://github.com/dutch-casa/fraud-benford.git"

if "google.colab" in sys.modules:
    if not os.path.exists("fraud-benford"):
        !git clone {REPO_URL}
    %cd fraud-benford
    !pip install -q -r requirements.txt

repo_root = os.path.abspath("." if os.path.basename(os.getcwd()) == "fraud-benford" else "..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

In [ ]:
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.data import load_raw, time_ordered_three_way_split
from src.benford import (
    BENFORD_PROBS,
    benford_features,
    chi_square_distance,
    empirical_distribution,
    leading_digit_series,
)
from src.features import build_feature_matrix
from src.evaluation import (
    EvalReport,
    evaluate,
    model_comparison_table,
    plot_benford_histogram,
    plot_pr_curves,
    plot_roc_curves,
)
from src.models import (
    autoencoder_anomaly,
    lightgbm_classifier,
    logistic_regression,
    mlp_classifier,
    xgboost_classifier,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = [10, 6]
RANDOM_SEED = 42

## 1. Problem Definition and Scope

Detect fraudulent credit card transactions on the ULB public dataset (284,807 transactions over ~48 hours, 492 labeled as fraud — a 0.17% positive rate).

**Objectives (SMART):**
- **Specific:** binary classification of fraud vs. legitimate transactions.
- **Measurable:** area under the precision-recall curve (AUPRC), precision at recall ≥ 0.80.
- **Achievable:** published baselines on this dataset report AUPRC ≈ 0.75 for strong models.
- **Relevant:** mirrors the real-world task Stripe Radar and similar systems solve.
- **Time-bound:** one Colab Run-All session.

**Approach:** staircase progression from a weak linear baseline to gradient-boosted trees, a small MLP, and a torch autoencoder, with Benford's Law features bolted on at one step to measure their marginal contribution. Gradient-boosted trees are additionally tuned on a held-out validation slice and ablated against a base-features-only variant to isolate the contribution of the engineered features.

## 2. Data Loading and Exploration

In [ ]:
df = load_raw()
print(f"shape: {df.shape}")
print(f"fraud rate: {df['Class'].mean():.4%}")
print(f"time range: {df['Time'].min():.0f}s \u2192 {df['Time'].max():.0f}s ({df['Time'].max() / 3600:.1f} hours)")
df.head()

In [ ]:
n_dupes = int(df.duplicated().sum())
n_null = int(df.isna().sum().sum())
print(f"duplicate rows: {n_dupes} ({n_dupes / len(df):.2%})")
print(f"null values (all columns): {n_null}")
print("\nDecision: we keep duplicates. In a payments context a repeated charge can itself be a fraud signal (retries, card-testing), so dropping them would erase information the models may learn from. The report discusses this explicitly.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

counts = df["Class"].value_counts()
axes[0].bar(["legit", "fraud"], counts.values, color=["#4c72b0", "#c44e52"])
axes[0].set_yscale("log")
axes[0].set_title("Class distribution (log scale)")
axes[0].set_ylabel("Count")

axes[1].hist(df.loc[df["Class"] == 0, "Amount"], bins=60, alpha=0.6, label="legit", log=True)
axes[1].hist(df.loc[df["Class"] == 1, "Amount"], bins=60, alpha=0.8, label="fraud", log=True, color="#c44e52")
axes[1].set_title("Amount distribution")
axes[1].set_xlabel("Amount")
axes[1].set_ylabel("Count (log)")
axes[1].legend()

axes[2].hist(df.loc[df["Class"] == 0, "Time"] / 3600, bins=48, alpha=0.6, label="legit")
axes[2].hist(df.loc[df["Class"] == 1, "Time"] / 3600, bins=48, alpha=0.8, label="fraud", color="#c44e52")
axes[2].set_title("Transactions over time")
axes[2].set_xlabel("Hours since first transaction")
axes[2].set_ylabel("Count")
axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Preprocessing and Feature Engineering

### 3a. Benford's Law

Benford's Law says the leading digit $d$ of naturally-occurring numerical data follows $P(d) = \log_{10}(1 + 1/d)$ — digit 1 appears ~30% of the time, digit 9 only ~4.6%. Real transaction amounts tend to obey it. Fabricated or anomalous amounts often don't. We use this as both a diagnostic plot and a feature.

In [ ]:
digits_all = leading_digit_series(df["Amount"])
digits_legit = leading_digit_series(df.loc[df["Class"] == 0, "Amount"])
digits_fraud = leading_digit_series(df.loc[df["Class"] == 1, "Amount"])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_benford_histogram(digits_all, "All transactions", ax=axes[0])
plot_benford_histogram(digits_legit, "Legitimate", ax=axes[1])
plot_benford_histogram(digits_fraud, "Fraud", ax=axes[2])
plt.tight_layout()
plt.show()

for label, digits in [("all", digits_all), ("legit", digits_legit), ("fraud", digits_fraud)]:
    chi2 = chi_square_distance(empirical_distribution(digits))
    print(f"chi-square distance from Benford ({label}): {chi2:.4f}")

### 3b. Time-window aggregates

Inspired by Stripe's observation that card-testing attacks are bursty and sequential. We build rolling-window features over the `Time` axis: transaction counts in 60s/600s windows, rolling amount sums, time-since-last, and a rolling amount z-score.

**Design note on temporal locality:** these features are computed on the *entire* dataframe before the time-ordered split. That means the first rows of the val and test sets see history from the end of the train set. This is not a leakage bug — it mirrors a production system at inference time, where recent transaction history is always available when a new charge arrives. We flag it here as a deliberate choice and revisit it in the report.

In [ ]:
time_features = build_feature_matrix(df)
benford_feats = benford_features(df)
print(f"time-window feature shape: {time_features.shape}")
print(f"benford feature shape:     {benford_feats.shape}")
time_features.head()

## 4. Model Development

We split the data **time-ordered** three ways: 70% train, 10% validation, 20% test. The validation slice exists for hyperparameter tuning; the test slice is only touched once for the final comparison. No random shuffle — that would leak the future into the past.

### Feature sets

- **base** — the 28 PCA features + Amount (what every naive classifier gets)
- **+benford** — base plus 9 one-hot leading-digit features
- **+time** — base plus Benford plus 5 time-window aggregate features ("full")

In [ ]:
base_cols = [f"V{i}" for i in range(1, 29)] + ["Amount"]

features_base = df[base_cols].copy()
features_benford = pd.concat([features_base, benford_feats], axis=1)
features_full = pd.concat([features_benford, time_features], axis=1)
y = df["Class"].astype(int)

split = time_ordered_three_way_split(df, val_fraction=0.1, test_fraction=0.2)
train_end = len(split.train)
val_end = train_end + len(split.val)

print(f"train: {train_end:>6d} rows ({split.train['Class'].sum()} fraud)")
print(f"val:   {len(split.val):>6d} rows ({split.val['Class'].sum()} fraud)")
print(f"test:  {len(split.test):>6d} rows ({split.test['Class'].sum()} fraud)")

In [ ]:
def train_on_train_eval_on(
    model,
    X: pd.DataFrame,
    y: pd.Series,
    train_end: int,
    eval_start: int,
    eval_end: int | None = None,
) -> tuple[np.ndarray, EvalReport]:
    X_tr, y_tr = X.iloc[:train_end], y.iloc[:train_end]
    X_ev, y_ev = X.iloc[eval_start:eval_end], y.iloc[eval_start:eval_end]
    model.fit(X_tr, y_tr)
    scores = model.predict_proba(X_ev)[:, 1]
    return scores, evaluate(y_ev, scores)

def train_and_score(name, model, X, y, train_end, val_end):
    scores, report = train_on_train_eval_on(model, X, y, train_end, val_end, None)
    print(f"  {name:44s} AUPRC={report.auprc:.4f}  ROC-AUC={report.roc_auc:.4f}  P@R=0.80={report.precision_at_recall_80:.4f}")
    return scores, report

In [ ]:
n_train_frauds = int(y.iloc[:train_end].sum())
n_train_legit = train_end - n_train_frauds
spw = n_train_legit / max(n_train_frauds, 1)

experiments: list[tuple[str, object, pd.DataFrame]] = [
    ("1. LR (base)",                  logistic_regression(),                                             features_base),
    ("2. LR (base, balanced)",        logistic_regression(class_weight="balanced"),                     features_base),
    ("3. LR (+ Benford, balanced)",   logistic_regression(class_weight="balanced"),                     features_benford),
    ("4. LR (full, balanced)",        logistic_regression(class_weight="balanced"),                     features_full),
    ("5. XGBoost (full, untuned)",    xgboost_classifier(scale_pos_weight=spw),                          features_full),
    ("6. LightGBM (full, untuned)",   lightgbm_classifier(),                                              features_full),
    ("7. MLP (full, scaled)",         make_pipeline(StandardScaler(), mlp_classifier()),                 features_full),
    ("8. Autoencoder anomaly (full)", autoencoder_anomaly(),                                              features_full),
]

reports: dict[str, EvalReport] = {}
scored: dict[str, tuple[pd.Series, np.ndarray]] = {}
y_test = y.iloc[val_end:]

print("training baseline models...")
for name, model, feats in experiments:
    scores, reports[name] = train_and_score(name, model, feats, y, train_end, val_end)
    scored[name] = (y_test, scores)

### 4.5 Hyperparameter sweep on the validation slice

The baselines above use fixed reasonable defaults. We now sweep a small grid for LightGBM and XGBoost, selecting the best configuration **by AUPRC on the validation slice** and then evaluating the winner once on the test slice. The val slice is never used for evaluation of any baseline or winner — it exists solely for model selection.

In [ ]:
def sweep(name: str, factory, X: pd.DataFrame, grid: list[dict]) -> pd.DataFrame:
    rows = []
    for cfg in grid:
        model = factory(**cfg)
        _, report = train_on_train_eval_on(model, X, y, train_end, train_end, val_end)
        rows.append({**cfg, "AUPRC (val)": report.auprc, "P@R=0.80 (val)": report.precision_at_recall_80})
    return pd.DataFrame(rows).sort_values("AUPRC (val)", ascending=False).reset_index(drop=True)

In [ ]:
lgbm_grid = [
    {"n_estimators": n, "learning_rate": lr, "num_leaves": nl, "class_weight": cw}
    for n, lr, nl, cw in product([200, 500], [0.05, 0.1], [31, 127], [None, "balanced"])
]
print(f"sweeping {len(lgbm_grid)} LightGBM configurations...")
lgbm_results = sweep("LightGBM", lightgbm_classifier, features_full, lgbm_grid)
lgbm_results.head(10)

In [ ]:
xgb_grid = [
    {"n_estimators": n, "max_depth": md, "scale_pos_weight": spw_cfg, "learning_rate": 0.1}
    for n, md, spw_cfg in product([200, 500], [4, 6, 8], [1.0, 10.0, float(spw)])
]
print(f"sweeping {len(xgb_grid)} XGBoost configurations...")
xgb_results = sweep("XGBoost", xgboost_classifier, features_full, xgb_grid)
xgb_results.head(10)

In [ ]:
lgbm_best_cfg = {k: v for k, v in lgbm_results.iloc[0].items() if k not in ("AUPRC (val)", "P@R=0.80 (val)")}
xgb_best_cfg = {k: v for k, v in xgb_results.iloc[0].items() if k not in ("AUPRC (val)", "P@R=0.80 (val)")}

print("LightGBM best:", lgbm_best_cfg)
print("XGBoost best: ", xgb_best_cfg)

print("\nretraining winners on train, evaluating on test...")
name = "9. LightGBM (full, tuned)"
scores, reports[name] = train_and_score(name, lightgbm_classifier(**lgbm_best_cfg), features_full, y, train_end, val_end)
scored[name] = (y_test, scores)

name = "10. XGBoost (full, tuned)"
scores, reports[name] = train_and_score(name, xgboost_classifier(**xgb_best_cfg), features_full, y, train_end, val_end)
scored[name] = (y_test, scores)

### 4.6 Ablation: do engineered features help the tuned tree models?

The LR progression showed Benford and time-window features adding ~2 AUPRC points. Do gradient-boosted trees still benefit from them, or do trees pick up the same signal internally from V1–V28 and Amount? We train the tuned LightGBM configuration on `features_base` (no Benford, no time features) and compare against the same configuration on `features_full`.

In [ ]:
name = "11. LightGBM (base only, tuned)"
scores, reports[name] = train_and_score(name, lightgbm_classifier(**lgbm_best_cfg), features_base, y, train_end, val_end)
scored[name] = (y_test, scores)

name = "12. LightGBM (+ Benford, tuned)"
scores, reports[name] = train_and_score(name, lightgbm_classifier(**lgbm_best_cfg), features_benford, y, train_end, val_end)
scored[name] = (y_test, scores)

ablation = model_comparison_table({k: reports[k] for k in ["11. LightGBM (base only, tuned)", "12. LightGBM (+ Benford, tuned)", "9. LightGBM (full, tuned)"]})
print("\nablation (tuned LightGBM across feature sets):")
ablation

## 5. Evaluation

In [ ]:
comparison = model_comparison_table(reports)
comparison.style.format({"AUPRC": "{:.4f}", "ROC-AUC": "{:.4f}", "P@R=0.80": "{:.4f}", "Best F1": "{:.4f}", "Threshold": "{:.4f}"})

In [ ]:
plot_pr_curves(scored)

In [ ]:
plot_roc_curves(scored)

In [ ]:
best_name = comparison.iloc[0]["model"]
best = reports[best_name]
print(f"best model by AUPRC: {best_name}")
print(f"  AUPRC: {best.auprc:.4f}")
print(f"  ROC-AUC: {best.roc_auc:.4f}")
print(f"  best F1: {best.best_f1:.4f} at threshold {best.best_f1_threshold:.4f}")
print(f"  confusion matrix (rows=true, cols=pred, order=[legit, fraud]):")
print(best.confusion_at_best_f1)

## 6. Discussion and Limitations

### Findings

**Class weighting is the first big win.** Going from the naive LR baseline to `class_weight="balanced"` lifts AUPRC from 0.7186 to 0.7440 — a larger jump than any single feature change. On a 0.17% positive-rate dataset, telling the linear model that positives are rare is worth more than any new column.

**Engineered features help the linear model.** Benford's leading-digit features contribute a small but real ~0.004 AUPRC lift on top of balanced LR. Time-window aggregates contribute ~0.019 — an order of magnitude larger. Together they push LR from 0.7440 to 0.7667 without changing the model family. This is the clearest signal that the engineered features carry independent information.

**Gradient-boosted trees are the practical winner.** LightGBM (tuned) and MLP (scaled) are statistically indistinguishable on AUPRC, but LightGBM is better at every threshold that matters in production. At 80% recall, the tuned LightGBM's precision is substantially higher than the MLP's — which is the metric that decides how many false positives your fraud team has to review. A production system would pick the tree.

**Naive class weighting wrecks XGBoost calibration.** Untuned XGBoost with `scale_pos_weight` set to the raw class ratio (~500) produces a reasonable AUPRC (0.7796) but catastrophic precision at recall=0.80 (0.2521). The scores are saturated — good at *ranking* but useless at any fixed threshold. The hyperparameter sweep fixes this by discovering that a moderate positive-class weight (typically 10 or 1) is better for calibration. This is a reportable finding: the naive "balance the classes" heuristic is a trap in gradient boosting.

**The ablation shows engineered features help trees too, but by less.** The tuned LightGBM on `features_base` (V1–V28 + Amount only) versus `features_full` (base + Benford + time-window aggregates) shows a small but real gap. Trees can *partially* reconstruct the signal from raw features — that's what their splits are for — but the explicit feature engineering still helps at the margin. Benford's contribution is small even for trees; the time-window aggregates are where most of the lift comes from.

**The autoencoder underperforms dramatically.** AUPRC 0.1374 is ~6x worse than any supervised model. Reconstruction error, trained on legit-only data, is simply not the same signal as labeled fraud. Keeping it in the comparison gives us a lower bound and a point of contrast: unsupervised anomaly detection is *not* a drop-in replacement for supervised fraud detection, even when labels are rare.

**The Stripe connection — honestly.** Stripe's Payments Foundation Model claims big gains from sequence structure on card-testing attacks. Our time-window aggregate features are a weak approximation: we have no per-card grouping, so our windows aggregate globally. The modest lift we observed from these features (~0.019 AUPRC on LR, smaller on trees) is consistent with a weaker version of the same idea — but does not reproduce Stripe's magnitude of improvement, nor could it on this dataset without card identifiers.

### Limitations

- **PCA-anonymized features.** V1–V28 are already principal components, so domain-aware feature engineering is impossible. Any gains have to come from the modeling side or from the two features we can read: `Time` and `Amount`.
- **No per-card grouping.** The ULB dataset does not carry a card or user identifier, so we cannot reproduce Stripe's *per-entity sequence* signal. Our time-window features aggregate globally, which is strictly weaker.
- **Single dataset.** Results here do not generalize to modern payments fraud, which differs in attack patterns, amount distributions, and the ratio of card-testing to other fraud types. The ULB dataset is from 2013.
- **Single time-ordered 70/10/20 split, not rolling evaluation.** Real production systems retrain continuously and evaluate on rolling windows. Our test set is a single held-out window at the end of the 48-hour period.
- **Small hyperparameter grid.** We sweep 24 configurations for LightGBM and 18 for XGBoost. A production tuning run would use orders of magnitude more configurations plus cross-validation. Our sweep is proof-of-concept, not exhaustive.